In [0]:
# ============================================================================
# Bronze Layer Ingestion — Medallion Architecture
# -----------------------------------------------------------------------------
# This notebook ingests 10 raw source datasets "as-is" into Delta tables
# in the Bronze layer.  No joins, filters, deduplication, or transformations
# are applied — the data is preserved in its original form.  Standard audit
# metadata columns (_ingested_at, _source_file) are appended to every table.
#
# A reusable loop drives all 10 ingestions so that each table is processed
# identically and the code stays DRY.
# ============================================================================

from pyspark.sql.functions import current_timestamp, col, lit
from pyspark.sql.types import TimestampType

# ------------------------------------------------------------------
# Configuration — update these values for your environment
# ------------------------------------------------------------------
SOURCE_PATH    = "/Volumes/workspace/applebees/challenge_data/Applebees Challenge"   # root folder; each table lives in a sub-folder
SOURCE_FORMAT  = "json"                                     # csv | json | parquet | etc.
TARGET_CATALOG = "workspace"
TARGET_SCHEMA  = "bronze_schema"

# ------------------------------------------------------------------
# Table registry  (table_name → sub-directory under SOURCE_PATH)
# ------------------------------------------------------------------
bronze_tables = {
    "checks":                   "facts/checks",
    "checks_discounts":        "checks_discounts",
    "checks_items":             "checks_items",
    "checks_payments":          "checks_payments",
    "checks_tax":                "checks_tax",
    "checks_voids":             "checks_voids",
    "employees":                 "dimensions/employees",
    "items_table":               "dimensions/items",
    "locations_with_coordinates": "dimensions/locations",
    "revenue_centers":            "dimensions/revenue_centers",
}

# ------------------------------------------------------------------
# Format-specific reader options  (extend as needed)
# ------------------------------------------------------------------
format_options = {
    "csv":    {"header": "true", "inferSchema": "true"},
    "json":   {"multiLine": "true"},
    "parquet": {},
}

# ------------------------------------------------------------------
# Ensure the target schema exists
# ------------------------------------------------------------------
spark.sql(f"USE CATALOG {TARGET_CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

# ------------------------------------------------------------------
# Reusable ingestion function (batch / one-time load)
# ------------------------------------------------------------------
def ingest_to_bronze(table_name: str, sub_dir: str, source_format: str):
    """
    Read raw files from SOURCE_PATH/<sub_dir> and append them as-is into a
    Delta table named TARGET_CATALOG.TARGET_SCHEMA.<table_name>.

    Adds two metadata columns:
        _ingested_at  – timestamp when the row was processed
        _source_file  – path of the source file the row came from
    """
    full_source_path = f"{SOURCE_PATH}/{sub_dir}"
    full_table_name  = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"

    print(f"▶ Ingesting {full_source_path}  →  {full_table_name}")

    # --- Read raw data as-is ------------------------------------------------
    reader = (spark.read
              .format(source_format)
              .options(**format_options.get(source_format, {})))

    df = reader.load(full_source_path)

    # --- Append auditability metadata columns -------------------------------
    df = (df
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path")))

    # --- Write to Delta (create table on first run, append on subsequent) ---
    (df.write
       .format("delta")
       .mode("overwrite")          # first load creates the table; switch to "append" for incremental
       .saveAsTable(full_table_name))

    row_count = spark.table(full_table_name).count()
    print(f"  ✅ {full_table_name} created  —  {row_count:,} rows\n")
    return full_table_name, row_count

# ------------------------------------------------------------------
# Execute ingestion for all 10 tables
# ------------------------------------------------------------------
results = []
for table_name, sub_dir in bronze_tables.items():
    try:
        full_name, rows = ingest_to_bronze(table_name, sub_dir, SOURCE_FORMAT)
        results.append((full_name, rows, "OK"))
    except Exception as e:
        print(f"  ❌ {table_name} FAILED: {e}\n")
        results.append((f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}", 0, str(e)))

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print("=" * 70)
print("Bronze ingestion summary")
print("=" * 70)
for full_name, rows, status in results:
    print(f"  {full_name:<55} {rows:>10,}  {status}")
print("=" * 70)

# ------------------------------------------------------------------
# OPTIONAL — Auto Loader (spark.readStream) version for incremental/streaming
# ------------------------------------------------------------------
# Uncomment the block below if you prefer continuous incremental ingestion.
# Each table will be written by a separate streaming query using
# `cloudFiles` (Auto Loader) and Delta `merge` / `append`.
#
# def stream_to_bronze(table_name: str, sub_dir: str, source_format: str):
#     full_source_path = f"{SOURCE_PATH}/{sub_dir}"
#     full_table_name  = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"
#
#     stream_df = (spark.readStream
#         .format("cloudFiles")
#         .option("cloudFiles.format", source_format)
#         .option("cloudFiles.schemaLocation", f"{SOURCE_PATH}/_schemas/{table_name}")
#         .option("cloudFiles.inferColumnTypes", "true")
#         .load(full_source_path)
#         .withColumn("_ingested_at", current_timestamp())
#         .withColumn("_source_file", input_file_name()))
#
#     (stream_df.writeStream
#         .format("delta")
#         .outputMode("append")
#         .option("checkpointLocation", f"{SOURCE_PATH}/_checkpoints/{table_name}")
#         .option("mergeSchema", "true")
#         .toTable(full_table_name))
#
# for table_name, sub_dir in bronze_tables.items():
#     stream_to_bronze(table_name, sub_dir, SOURCE_FORMAT)
#
# # Keep streaming queries alive in a notebook:
# # spark.streams.awaitAnyTermination()

In [0]:
# ------------------------------------------------------------------
# Explode array-type columns from the checks Bronze table into
# individual Bronze tables:
#   checks_discounts, checks_items, checks_payments, checks_tax, checks_voids
# ------------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, col, explode, lit
from pyspark.sql.types import ArrayType, StructType

CHECKS_TABLE       = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.checks"

# Desired target table names (order preserved)
target_table_names = [
    "checks_discounts",
    "checks_items",
    "checks_payments",
    "checks_tax",
    "checks_voids",
]

# ------------------------------------------------------------------
# Inspect the checks table schema and find array-type columns
# ------------------------------------------------------------------
checks_df     = spark.table(CHECKS_TABLE)
schema_fields = {f.name: f.dataType for f in checks_df.schema.fields}
array_cols    = {name for name, dtype in schema_fields.items() if isinstance(dtype, ArrayType)}

print(f"Array columns detected in {CHECKS_TABLE}: {sorted(array_cols)}")

# Map each array column to its target table name
# (e.g. column "discounts" → table "checks_discounts")
column_to_table = {}
for tbl in target_table_names:
    col_guess = tbl.replace("checks_", "", 1)          # "discounts", "items", …
    match = next((ac for ac in array_cols if ac.lower() == col_guess.lower()), None)
    if match is None:
        # Fall back to known column-name overrides (camelCase / custom names)
        overrides = {"checks_items": "ItemsSold"}
        override_col = overrides.get(tbl)
        if override_col and override_col.lower() in {ac.lower() for ac in array_cols}:
            match = next(ac for ac in array_cols if ac.lower() == override_col.lower())
    if match:
        column_to_table[match] = tbl
    else:
        print(f"⚠ No array column matching '{col_guess}' found — '{tbl}' will be skipped")

print(f"Mapping: {column_to_table}\n")

# Identify a check-level key column for traceability
check_key = next((c for c in ["check_id", "checkId", "id", "_id", "check_number"]
                  if c in checks_df.columns), None)

# ------------------------------------------------------------------
# Explode each array column and write its own Delta table
# ------------------------------------------------------------------
results = []
for array_col, table_name in column_to_table.items():
    full_table_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"
    print(f"▶ Exploding '{array_col}'  →  {full_table_name}")

    # Select the check key (if found) + original source file + the array column
    select_exprs = []
    if check_key:
        select_exprs.append(col(check_key))
    if "_source_file" in checks_df.columns:
        select_exprs.append(col("_source_file"))
    select_exprs.append(col(array_col))

    exploded_df = (checks_df
                   .select(*select_exprs)
                   .withColumn(array_col, explode(col(array_col))))

    # Flatten struct fields into top-level columns (if array elements are structs)
    elem_type = schema_fields[array_col].elementType
    if isinstance(elem_type, StructType):
        for sub_field in elem_type.fields:
            exploded_df = exploded_df.withColumn(sub_field.name, col(f"{array_col}.{sub_field.name}"))
        exploded_df = exploded_df.drop(array_col)
    else:
        # Non-struct array: keep the exploded value with a meaningful name
        exploded_df = exploded_df.withColumnRenamed(array_col, f"{array_col}_value")

    # Add / refresh audit metadata
    exploded_df = exploded_df.withColumn("_ingested_at", current_timestamp())
    if "_source_file" not in exploded_df.columns:
        exploded_df = exploded_df.withColumn("_source_file", lit(CHECKS_TABLE))

    # Write to Delta
    (exploded_df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(full_table_name))

    row_count = spark.table(full_table_name).count()
    print(f"  ✅ {full_table_name} created  —  {row_count:,} rows\n")
    results.append((full_table_name, row_count, "OK"))

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print("=" * 70)
print("Array explosion summary")
print("=" * 70)
for full_name, rows, status in results:
    print(f"  {full_name:<55} {rows:>10,}  {status}")
print("=" * 70)

In [0]:
# ------------------------------------------------------------------
# Preview all 10 Bronze tables
# ------------------------------------------------------------------
bronze_preview_tables = [
    "checks",
    "checks_discounts",
    "checks_items",
    "checks_payments",
    "checks_tax",
    "checks_voids",
    "employees",
    "items_table",
    "locations_with_coordinates",
    "revenue_centers",
]

for tbl in bronze_preview_tables:
    full_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{tbl}"
    try:
        print(f"▶ {full_name}  ({spark.table(full_name).count():,} rows)")
        display(spark.table(full_name).limit(10))
        print("=" * 70)
    except Exception:
        print(f"▶ {full_name}  (table not found — skipped)")
        print("=" * 70)

In [0]:
display(spark.sql("SHOW TABLES IN workspace.bronze_schema"))

In [0]:
spark.table("workspace.bronze_schema.checks").printSchema()